# Prática - Módulo 15: Análise Exploratória de Dados (EDA) — Previsão de Churn

Este notebook contém o fluxo completo de Análise Univariada, Bivariada, detecção de outliers e identificação das principais variáveis preditoras para o **Churn**.

In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


pd.set_option('display.max_columns', None)

In [2]:

df = pd.read_csv('CHURN_TELECON_MOD08_TAREFA.csv', delimiter=';')

colunas_remover = [col for col in ['Unnamed: 0', 'customerID'] if col in df.columns]
df.drop(columns=colunas_remover, inplace=True)

df.head()

,Genero,Idoso,Casado,Dependents,Tempo_como_Cliente,PhoneService,Servico_Internet,Servico_Seguranca,Suporte_Tecnico,StreamingTV,Tipo_Contrato,PaymentMethod,Pagamento_Mensal,Total_Pago,Churn
0,Female,0,Yes,No,1,No,DSL,No,No,No,Month-to-month,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,DSL,Yes,No,No,One year,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,DSL,Yes,No,No,Month-to-month,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,DSL,Yes,Yes,No,One year,Bank transfer (automatic),NaN,1840.75,No
4,NaN,0,No,No,2,Yes,Fiber optic,No,No,No,Month-to-month,Electronic check,NaN,151.65,Yes


## 3. Análise Univariada & Detecção de Outliers

In [3]:

display(df.describe())

colunas_numericas = ['Tempo_como_Cliente', 'Pagamento_Mensal', 'Total_Pago']
colunas_numericas = [c for c in colunas_numericas if c in df.columns]

for coluna in colunas_numericas:
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]
    print(f"{coluna}: {len(outliers)} outliers detectados (Limites: {limite_inferior:.2f} a {limite_superior:.2f})")

    fig = px.box(df, x=coluna, title=f'<b>Distribuição e Outliers: {coluna}</b>')
    if coluna != 'Tempo_como_Cliente':
        fig.update_xaxes(tickprefix='R$ ')
    fig.show()

,Idoso,Tempo_como_Cliente,Pagamento_Mensal,Total_Pago
count,2500.00000,2500.000000,2175.000000,2500.000000
mean,0.16080,32.341600,65.607563,2291.882880
std,0.36742,24.626398,29.931520,2265.347752
min,0.00000,0.000000,18.400000,18.800000
25%,0.00000,8.000000,39.500000,402.337500
50%,0.00000,29.000000,71.450000,1405.325000
75%,0.00000,56.000000,90.250000,3867.162500
max,1.00000,72.000000,118.650000,8564.750000


Tempo_como_Cliente: 0 outliers detectados (Limites: -64.00 a 128.00)


Pagamento_Mensal: 0 outliers detectados (Limites: -36.62 a 166.38)


Total_Pago: 0 outliers detectados (Limites: -4794.90 a 9064.40)


In [4]:

fig_churn = px.histogram(
    df, 
    x='Churn', 
    color='Churn', 
    text_auto=True,
    title='<b>Distribuição da Variável Alvo (Churn)</b>'
)
fig_churn.show()
print("Proporção de Churn (%):")
print((df['Churn'].value_counts(normalize=True) * 100).round(2))

Proporção de Churn (%):
Churn
No     73.99
Yes    26.01
Name: proportion, dtype: float64


## 4A. Análise Bivariada (5 Questionamentos focados no negócio e no Churn)

In [5]:
# Questionamento 1: Quais são os serviços adicionais mais contratados pelos clientes?
servicos = ['Servico_Telefone', 'Multiplas_Linhas', 'Servico_Internet', 'Seguranca_Online', 
            'Backup_Online', 'Protecao_Dispositivo', 'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes']
servicos_existentes = [s for s in servicos if s in df.columns]

contagem_servicos = {s: (df[s] == 'Sim').sum() if df[s].dtype == 'object' else (df[s] == 1).sum() for s in servicos_existentes}
df_servicos = pd.DataFrame(list(contagem_servicos.items()), columns=['Servico', 'Total_Clientes']).sort_values(by='Total_Clientes', ascending=False)

fig_q1 = px.bar(df_servicos, x='Servico', y='Total_Clientes', text_auto=True, title='<b>Q1: Adesão aos Serviços Adicionais</b>')
fig_q1.show()

In [6]:
# Questionamento 2: Clientes com mais serviços contratados têm menor taxa de Churn?
df['Qtd_Servicos'] = sum((df[s] == 'Sim').astype(int) if df[s].dtype == 'object' else (df[s] == 1).astype(int) for s in servicos_existentes)

fig_q2 = px.histogram(
    df, 
    x='Qtd_Servicos', 
    color='Churn', 
    barmode='group', 
    text_auto=True,
    title='<b>Q2: Quantidade de Serviços Contratados vs. Churn</b>'
)
fig_q2.show()

In [7]:
# Questionamento 3: O perfil familiar (Casado / Dependentes) influencia o cancelamento?
for col_fam in ['Casado', 'Dependentes']:
    if col_fam in df.columns:
        fig_q3 = px.histogram(
            df, 
            x=col_fam, 
            color='Churn', 
            barmode='group', 
            text_auto=True,
            title=f'<b>Q3: Relação entre {col_fam} e Churn</b>'
        )
        fig_q3.show()

In [8]:
# Questionamento 4: Como o Tempo como Cliente e o Pagamento Mensal se comportam em relação ao Churn?
tabela_medias = df.groupby('Churn')[colunas_numericas].mean().round(2)
display(tabela_medias)

fig_q4 = px.box(
    df, 
    x='Churn', 
    y='Tempo_como_Cliente', 
    color='Churn',
    title='<b>Q4: Distribuição do Tempo como Cliente por Status de Churn</b>'
)
fig_q4.show()

,Tempo_como_Cliente,Pagamento_Mensal,Total_Pago
Churn,,,
No,37.41,61.72,2549.33
Yes,17.98,76.57,1562.46


In [9]:
# Questionamento 5: Qual o impacto do Tipo de Contrato e do Serviço de Internet na taxa de Churn?
for col_cat in ['Tipo_Contrato', 'Servico_Internet']:
    if col_cat in df.columns:
        fig_q5 = px.histogram(
            df, 
            x=col_cat, 
            color='Churn', 
            barmode='group',
            text_auto=True,
            title=f'<b>Q5: Relação entre {col_cat} e Churn</b>'
        )
        fig_q5.show()

## 4B. Seleção e Importância das Variáveis para o Churn

In [10]:

df_encoded = df.copy()
le = LabelEncoder()

for col in df_encoded.select_dtypes(include=['object', 'category']).columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

correlacao_churn = df_encoded.corr()['Churn'].sort_values(ascending=False)
print("Correlação das variáveis com o Churn:")
print(correlacao_churn)

X = df_encoded.drop(columns=['Churn', 'Qtd_Servicos'], errors='ignore')
y = df_encoded['Churn']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importancias = pd.DataFrame({
    'Variavel': X.columns,
    'Importancia': rf.feature_importances_
}).sort_values(by='Importancia', ascending=False)

fig_imp = px.bar(
    importancias.head(10), 
    x='Importancia', 
    y='Variavel', 
    orientation='h',
    title='<b>Top 10 Variáveis Mais Importantes para Previsão de Churn (Random Forest)</b>'
)
fig_imp.update_layout(yaxis={'categoryorder':'total ascending'})
fig_imp.show()

C:\Users\tec01\AppData\Local\Temp\ipykernel_9680\316326491.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_encoded.select_dtypes(include=['object', 'category']).columns:


Correlação das variáveis com o Churn:
Churn                 1.000000
Pagamento_Mensal      0.218227
Idoso                 0.122400
PaymentMethod         0.095939
PhoneService          0.002042
Genero               -0.001748
StreamingTV          -0.035403
Servico_Internet     -0.042006
Casado               -0.149609
Dependents           -0.169220
Total_Pago           -0.189306
Servico_Seguranca    -0.286670
Suporte_Tecnico      -0.312683
Tempo_como_Cliente   -0.342589
Tipo_Contrato        -0.406128
Qtd_Servicos               NaN
Name: Churn, dtype: float64


### Conclusão da Etapa 4B:
Com base na análise exploratória, na correlação e na importância extraída pelo modelo Random Forest, as variáveis mais relevantes para prever o **Churn** são:
1. **`Tempo_como_Cliente`**: Clientes mais recentes possuem probabilidade significativamente maior de cancelar.
2. **`Tipo_Contrato`**: Contratos mensais (*Month-to-month*) concentram a maior parte das evasões por não exigirem fidelidade.
3. **`Pagamento_Mensal` e `Total_Pago`**: Faturas mensais mais elevadas aumentam a sensibilidade ao preço e o risco de cancelamento.
4. **`Servico_Internet` e `Suporte_Tecnico`**: Clientes com fibra óptica sem suporte técnico ou segurança adicional apresentam maior taxa de churn.